In [1]:
import requests
from PIL import Image
from io import BytesIO
from bs4 import BeautifulSoup
from transformers import AutoProcessor, BlipForConditionalGeneration

In [ ]:
processor = AutoProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

In [13]:
url = "https://en.wikipedia.org/wiki/New_York_City"

headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
print("Status:", response.status_code, "HTML size:", len(response.text))

soup = BeautifulSoup(response.text, "html.parser")
img_elements = soup.find_all("img")
print(f"Found {len(img_elements)} <img> tags")

with open("captions.txt", "w", encoding="utf-8") as caption_file:
    for idx, img_element in enumerate(img_elements, start=1):
        img_url = img_element.get("src") or img_element.get("data-src")
        if not img_url and img_element.has_attr("srcset"):
            img_url = img_element["srcset"].split()[0]
        if not img_url:
            continue

        if img_url.endswith(".svg") or ".svg" in img_url:
            continue

        if img_url.startswith("//"):
            img_url = "https:" + img_url
        elif img_url.startswith("/"):
            img_url = "https://en.wikipedia.org" + img_url
        elif not img_url.startswith("http"):
            continue

        try:
            r = requests.get(img_url, timeout=10, headers=headers)
            raw_image = Image.open(BytesIO(r.content))

            if raw_image.size[0] * raw_image.size[1] < 200:
                continue

            raw_image = raw_image.convert("RGB")

            text = "the image of"
            inputs = processor(images=raw_image, text=text, return_tensors="pt")
            out = model.generate(**inputs, max_new_tokens=50)
            caption = processor.decode(out[0], skip_special_tokens=True)

            caption_file.write(f"{img_url}: {caption}\n")
            print(f"[{idx}] Caption saved")

        except OSError:
            continue
        except Exception as e:
            print(f"[{idx}] Error: {e}")
            continue

Status: 200 HTML size: 1652814
Found 139 <img> tags
[5] Caption saved
[6] Caption saved
[7] Caption saved
[8] Caption saved
[9] Caption saved
[10] Caption saved
[11] Caption saved
[12] Caption saved
[13] Caption saved
[17] Caption saved
[20] Caption saved
[25] Caption saved
[26] Caption saved
[27] Caption saved
[28] Caption saved
[29] Caption saved
[30] Caption saved
[31] Caption saved
[32] Caption saved
[33] Caption saved
[34] Caption saved
[35] Caption saved
[36] Caption saved
[37] Caption saved
[38] Caption saved
[40] Caption saved
[41] Caption saved
[42] Caption saved
[43] Caption saved
[44] Caption saved
[45] Caption saved
[46] Caption saved
[47] Caption saved
[48] Caption saved
[49] Caption saved
[50] Caption saved
[51] Caption saved
[52] Caption saved
[53] Caption saved
[54] Caption saved
[55] Caption saved
[56] Caption saved
[57] Caption saved
[58] Caption saved
[59] Caption saved
[60] Caption saved
[61] Caption saved
[62] Caption saved
[63] Caption saved
[64] Caption saved
[65

In [14]:
with open("captions.txt", "r", encoding="utf-8") as caption_file:
    for line in caption_file:
        print(line.strip())

https://upload.wikimedia.org/wikipedia/commons/thumb/7/7a/View_of_Empire_State_Building_from_Rockefeller_Center_New_York_City_dllu_%28cropped%29.jpg/330px-View_of_Empire_State_Building_from_Rockefeller_Center_New_York_City_dllu_%28cropped%29.jpg: the image of new york, new york, and manhattan
https://upload.wikimedia.org/wikipedia/commons/thumb/0/0f/67%C2%BA_Per%C3%ADodo_de_Sesiones_de_la_Asamblea_General_de_Naciones_Unidas_%288020913157%29_%28cropped%29.jpg/120px-67%C2%BA_Per%C3%ADodo_de_Sesiones_de_la_Asamblea_General_de_Naciones_Unidas_%288020913157%29_%28cropped%29.jpg: the image of a city street
https://upload.wikimedia.org/wikipedia/commons/thumb/0/0e/Statue-of-Liberty-New-York-2014.jpg/120px-Statue-of-Liberty-New-York-2014.jpg: the image of the statue of liberty
https://upload.wikimedia.org/wikipedia/commons/thumb/b/ba/New_york_times_square-terabass_%28cropped%29.jpg/120px-New_york_times_square-terabass_%28cropped%29.jpg: the image of times square in new york
https://upload.wiki